# Generador de dataset: frase chilena → frase neutralizada

Para cada expresión de `rag_documentos.jsonl` (con sus definiciones y ejemplos) se le pide a **Mistral-7B-Instruct** `K` pares `(frase chilena, frase neutralizada)`. El resultado se guarda en un CSV.

**Flujo:** leer el jsonl → armar un prompt por expresión → generar en lotes en la GPU → parsear el JSON que devuelve el modelo → validar y deduplicar → guardar de a poco (se puede interrumpir y retomar).

```
pip install torch --index-url https://download.pytorch.org/whl/cu128
pip install -U transformers accelerate bitsandbytes pandas tqdm
```

Mistral-7B-Instruct es un modelo con acceso restringido en Hugging Face: hay que aceptar sus términos en la página del modelo y luego correr `huggingface-cli login` (o `from huggingface_hub import login; login()`).

In [1]:
import os
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

In [2]:
%pip install -U accelerate bitsandbytes ipywidgets

   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   --------- ------------------------------ 0.5/2.2 MB 2.4 MB/s eta 0:00:01
   ------------------ --------------------- 1.0/2.2 MB 2.5 MB/s eta 0:00:01
   ---------------------------- ----------- 1.6/2.2 MB 2.6 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 2.7 MB/s  0:00:00

   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   ---------------------------------------- 3/3 [ipywidgets]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# [GPU] Verificación del entorno
import torch

print("torch:", torch.__version__, "| CUDA del build:", torch.version.cuda, "| disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} | {p.total_memory / 1e9:.1f} GB | sm_{p.major}{p.minor}")
else:
    print("No hay GPU visible: revisa que torch sea el build con CUDA 12.8 (cu128).")

torch: 2.14.0+cu130 | CUDA del build: 13.0 | disponible: True
GPU: NVIDIA GeForce RTX 5070 | 12.8 GB | sm_120


In [4]:
from pathlib import Path

def encontrar_raiz(archivo="scraping/data/rag_documentos.jsonl"):
    """Sube desde la carpeta actual hasta encontrar la carpeta del proyecto que contiene el jsonl."""
    actual = Path.cwd().resolve()
    for carpeta in [actual, *actual.parents]:
        if (carpeta / archivo).is_file():
            return carpeta
    raise FileNotFoundError(f"No encontré {archivo} subiendo desde {actual}")

RAIZ = encontrar_raiz()
RUTA_ENTRADA = RAIZ / "scraping/data/rag_documentos.jsonl"
RUTA_SALIDA = RAIZ / "data/entrenamiento/dataset_chilenismos.csv"
RUTA_FILTRADO = RAIZ / "data/entrenamiento/dataset_chilenismos_filtrado.csv"
RUTA_SALIDA.parent.mkdir(parents=True, exist_ok=True)

In [5]:
# Configuración
from pathlib import Path

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"
K = 5             # pares (chilena, neutral) que se piden por expresión
MAX_DEFS = 4      # definiciones que se muestran en el prompt (las de más votos)
MAX_RONDAS = 3    # veces que se reintenta una expresión si faltan pares válidos
BATCH = 8         # expresiones por llamada a la GPU
TEMPERATURA = 0.8
TOP_P = 0.9
SEED = 42
LIMITE = None     # p. ej. 20 para una prueba rápida; None = todas las expresiones

In [6]:
# Lectura del jsonl
import json
import pandas as pd


def cargar_docs(ruta):
    docs = []
    with open(ruta, encoding="utf-8") as f:
        for i, linea in enumerate(f, 1):
            linea = linea.strip()
            if not linea:
                continue
            try:
                docs.append(json.loads(linea))
            except json.JSONDecodeError:
                print(f"Línea {i} inválida, se omite")
    return docs


def tiene_definicion(doc):
    return any((x.get("definicion") or "").strip() for x in doc.get("definiciones", []))


docs_todos = cargar_docs(RUTA_ENTRADA)
docs = [d for d in docs_todos if tiene_definicion(d)]
n_defs = [len(d["definiciones"]) for d in docs]
print(f"Expresiones leídas: {len(docs_todos)} | con al menos una definición: {len(docs)}")
print(f"Definiciones por expresión: media {sum(n_defs) / len(n_defs):.1f}, máx. {max(n_defs)}")
print(f"Marcadas como completas: {sum(bool(d.get('completo')) for d in docs)}")

Expresiones leídas: 802 | con al menos una definición: 802
Definiciones por expresión: media 2.0, máx. 14
Marcadas como completas: 802


In [7]:
# [GPU] Carga del modelo en 4 bits (NF4)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed

set_seed(SEED)
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token     # Mistral no trae token de relleno
tokenizer.padding_side = "left"               # para generar en lotes el relleno va a la izquierda
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map={"": 0}, attn_implementation="sdpa"
)
model.eval()
print(f"Memoria GPU usada: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

W1109 01:59:29.029000 24152 site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Memoria GPU usada: 4.1 GB


In [13]:
# Prompt, parser y validaciones (Python puro, no usa la GPU)
import re
import unicodedata

EJEMPLO_FORMATO = '[{"def": 1, "chilena": "Espérame, voy al tiro.", "neutral": "Espérame, voy enseguida."}]'

PLANTILLA = """Eres un lingüista experto en el español de Chile y en español neutro. Te doy una expresión chilena con sus definiciones y ejemplos de uso, tomados de un diccionario colaborativo.

EXPRESIÓN: {termino}
{variantes}DEFINICIONES:
{definiciones}

Tarea: escribe {k} pares de frases. En cada par:
- "chilena": una frase natural, como la diría un chileno, que use la expresión (puedes conjugarla o ajustar género y número si hace falta). Debe ser original: no copies los ejemplos.
- "neutral": la misma frase reescrita en español neutro y estándar, comprensible para cualquier hispanohablante: sin chilenismos ni coloquialismos locales y sin la expresión original, conservando el significado, la información y el tono.
- "def": el número de la definición que ilustra la frase.

Reglas:
- Las frases de un mismo par deben decir lo mismo; solo cambia el registro y el vocabulario.
- Varía el contexto, la persona gramatical y el largo (entre 8 y 25 palabras). Los {k} pares deben ser distintos entre sí.
- No uses comillas dobles dentro de las frases.
- Responde SOLO con un arreglo JSON, sin texto adicional, con este formato (ejemplo con otra expresión, no lo copies):
{formato}"""


def definiciones_para_prompt(doc):
    defs = [x for x in doc.get("definiciones", []) if (x.get("definicion") or "").strip()]

    def votos(x):
        try:
            return int(x.get("votos") or 0)
        except (TypeError, ValueError):
            return 0

    defs.sort(key=votos, reverse=True)
    return defs[:MAX_DEFS]


def construir_prompt(doc, k):
    defs = definiciones_para_prompt(doc)
    bloques = []
    for i, d in enumerate(defs, 1):
        txt = f"[{i}] {d['definicion'].strip()}"
        ej = (d.get("ejemplo") or "").strip()
        if ej:
            txt += f"\n    Ejemplo: {ej[:300]}"
        bloques.append(txt)
    otras = [v for v in doc.get("variantes", []) if v != doc["termino"]]
    variantes = f"VARIANTES: {', '.join(otras)}\n" if otras else ""
    return PLANTILLA.format(termino=doc["termino"], variantes=variantes,
                            definiciones="\n".join(bloques), k=k, formato=EJEMPLO_FORMATO)


def norm(s):
    """Minúsculas, sin tildes ni puntuación, para comparar textos."""
    s = unicodedata.normalize("NFD", str(s).lower())
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def formas_de(doc):
    return {norm(f) for f in [doc["clave"], doc["termino"], *doc.get("variantes", [])] if norm(f)}


def contiene_expresion(frase, doc, flexible=True):
    """True si la frase contiene la expresión o una variante.
    Modo flexible: acepta conjugaciones comparando los primeros 4 caracteres de cada palabra de la expresión."""
    f = norm(frase)
    for forma in formas_de(doc):
        if f" {forma} " in f" {f} ":
            return True
    if flexible:
        palabras = f.split()
        for forma in formas_de(doc):
            toks = [t for t in forma.split() if len(t) > 2]
            if toks and all(any(w.startswith(t[:4]) for w in palabras) for t in toks):
                return True
    return False


def extraer_objetos(texto):
    """Saca los objetos JSON de la respuesta del modelo, aunque venga con texto alrededor o cortada."""
    texto = texto.strip()
    objs = []
    ini, fin = texto.find("["), texto.rfind("]")
    if ini != -1 and fin > ini:
        try:
            data = json.loads(texto[ini:fin + 1])
            if isinstance(data, list):
                objs = [x for x in data if isinstance(x, dict)]
        except json.JSONDecodeError:
            pass
    if not objs:
        for m in re.finditer(r"\{[^{}]*\}", texto):
            try:
                o = json.loads(m.group(0))
            except json.JSONDecodeError:
                continue
            if isinstance(o, dict):
                objs.append(o)
    return objs


def validar_par(obj, doc, defs):
    """Devuelve la fila lista para guardar o None si el par no sirve."""
    ch = re.sub(r"\s+", " ", str(obj.get("chilena", ""))).strip()
    ne = re.sub(r"\s+", " ", str(obj.get("neutral", ""))).strip()
    if not ch or not ne:
        return None
    if not (3 <= len(ch.split()) <= 45 and 3 <= len(ne.split()) <= 45):
        return None
    if norm(ch) == norm(ne):
        return None
    # no aceptar copias de los ejemplos del diccionario
    nc = norm(ch)
    for d in doc.get("definiciones", []):
        ne_ej = norm(d.get("ejemplo") or "")
        if len(ne_ej.split()) >= 4 and (nc in ne_ej or ne_ej in nc):
            return None
    try:
        i = int(obj.get("def"))
        def_id = defs[i - 1].get("id") if 1 <= i <= len(defs) else None
    except (TypeError, ValueError):
        def_id = None
    return {
        "clave": doc["clave"],
        "termino": doc["termino"],
        "def_id": def_id,
        "frase_chilena": ch,
        "frase_neutralizada": ne,
        "contiene_clave": contiene_expresion(ch, doc, flexible=True),
        "neutral_sin_clave": not contiene_expresion(ne, doc, flexible=False),
    }

In [9]:
# [GPU] Generación en lotes
@torch.inference_mode()
def generar_textos(prompts, max_new_tokens):
    # Se agrega "[" al final del turno del asistente para que arranque directamente con el arreglo JSON
    textos = [
        tokenizer.apply_chat_template([{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True) + "["
        for p in prompts
    ]
    enc = tokenizer(textos, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
    out = model.generate(
        **enc, max_new_tokens=max_new_tokens, do_sample=True,
        temperature=TEMPERATURA, top_p=TOP_P, pad_token_id=tokenizer.pad_token_id,
    )
    nuevos = out[:, enc["input_ids"].shape[1]:]
    return ["[" + t for t in tokenizer.batch_decode(nuevos, skip_special_tokens=True)]

In [10]:
# Pipeline: generar, validar, deduplicar, guardar
from tqdm.auto import tqdm

vistos = set()      # frases chilenas ya generadas (normalizadas) para no repetir en todo el dataset
incompletas = []    # (clave, pares obtenidos)


def generar_para_lote(lote):
    """Devuelve {clave: [filas]} con hasta K pares válidos por expresión (reintenta hasta MAX_RONDAS)."""
    res = {d["clave"]: [] for d in lote}
    pendientes = list(lote)
    for ronda in range(1, MAX_RONDAS + 1):
        if not pendientes:
            break
        pedir = [K - len(res[d["clave"]]) + 2 for d in pendientes]
        prompts = [construir_prompt(d, k) for d, k in zip(pendientes, pedir)]
        salidas = generar_textos(prompts, max_new_tokens=80 * max(pedir) + 100)
        for d, texto in zip(pendientes, salidas):
            defs = definiciones_para_prompt(d)
            for obj in extraer_objetos(texto):
                if len(res[d["clave"]]) >= K:
                    break
                fila = validar_par(obj, d, defs)
                if fila is None or norm(fila["frase_chilena"]) in vistos:
                    continue
                vistos.add(norm(fila["frase_chilena"]))
                fila["ronda"] = ronda
                fila["modelo"] = MODEL_ID
                res[d["clave"]].append(fila)
        pendientes = [d for d in pendientes if len(res[d["clave"]]) < K]
    return res


def guardar(filas):
    if not filas:
        return
    existe = RUTA_SALIDA.exists()
    RUTA_SALIDA.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(filas).to_csv(RUTA_SALIDA, mode="a", header=not existe, index=False,
                               encoding="utf-8" if existe else "utf-8-sig")


def ejecutar():
    ya = set()
    if RUTA_SALIDA.exists():                       # retomar: saltar lo ya generado
        prev = pd.read_csv(RUTA_SALIDA, encoding="utf-8-sig")
        ya = set(prev["clave"])
        vistos.update(norm(x) for x in prev["frase_chilena"])
        print(f"Retomando: {len(ya)} expresiones ya procesadas")
    por_hacer = [d for d in docs if d["clave"] not in ya]
    if LIMITE:
        por_hacer = por_hacer[:LIMITE]
    print(f"Por procesar: {len(por_hacer)} expresiones, {K} pares cada una")
    for i in tqdm(range(0, len(por_hacer), BATCH)):
        res = generar_para_lote(por_hacer[i:i + BATCH])
        guardar([f for fs in res.values() for f in fs])
        incompletas.extend((c, len(fs)) for c, fs in res.items() if len(fs) < K)
    print(f"Listo. Expresiones con menos de {K} pares: {len(incompletas)}")

In [11]:
# [GPU] Ejecutar (con LIMITE = 20 en la configuración se hace una prueba rápida)
ejecutar()

Por procesar: 802 expresiones, 5 pares cada una


  0%|          | 0/101 [00:00<?, ?it/s]

Listo. Expresiones con menos de 5 pares: 0


In [12]:
# Revisión del resultado y exportación del dataset filtrado
df = pd.read_csv(RUTA_SALIDA, encoding="utf-8-sig")
print(f"Pares generados: {len(df)} | expresiones: {df['clave'].nunique()}")
print(f"Pares por expresión: media {df.groupby('clave').size().mean():.2f}")
print(f"contiene_clave: {df['contiene_clave'].mean():.1%} | neutral_sin_clave: {df['neutral_sin_clave'].mean():.1%}")

filtrado = df[df["contiene_clave"] & df["neutral_sin_clave"]]
filtrado.to_csv(RUTA_FILTRADO, index=False, encoding="utf-8-sig")
print(f"Pares que pasan ambos chequeos: {len(filtrado)} ({len(filtrado) / len(df):.1%}) -> {RUTA_FILTRADO}")

pd.set_option("display.max_colwidth", None)
df.sample(min(10, len(df)), random_state=0)[["termino", "frase_chilena", "frase_neutralizada"]]

Pares generados: 4010 | expresiones: 802
Pares por expresión: media 5.00
contiene_clave: 77.5% | neutral_sin_clave: 99.6%
Pares que pasan ambos chequeos: 3093 (77.1%) -> C:\Users\conso\OneDrive\Desktop\Universidah\2026-2\NLP\Proyecto\NLP-ChiNeutro\data\entrenamiento\dataset_chilenismos_filtrado.csv


,termino,frase_chilena,frase_neutralizada
1098,ser pollo,"Mire, ese estudiante pollo no se está presentando a la clase.","Mire, ese estudiante no se está presentando a la clase."
2408,Andai de la perra,"No me puedes hablar así, andai de la perra, ¿no veo que te estoy ayudando?.","No puedes hablarme así, ¿no veo que estoy ayudándote?"
2436,a la chucha,"Tu abuela es tan lejos de aquí, que no podríamos llamar a la guardería.",Tu abuela está tan lejos que no podríamos llamar a la guardería.
3283,fierro pelao,"Tú que siempre eres tan cautelo, ¿por qué no lo hiciste antes?","Tú que siempre eres tan cauteloso, ¿por qué no lo hiciste antes?"
1366,Techecas,La nena le dio un bofetazo a su techeca.,La niña le dio un paliza a su espalda.
2252,piojo resucitado,"Ese policía que te acusó hace años, es un piojo resucitado","Ese policía que te acusó hace años, es una persona que olvida de su pasado"
2696,Andar paveando,"No le hables paveando a la abuela, ella se enferma fácil.","No le hables sin atención a la abuela, ella se enferma fácil."
2656,Aguja,"No te dejes llevar en la ciudad, hay que andar aguja.","No te dejes llevar en la ciudad, hay que andar con atención."
2188,Piante,"Sos piante, ¿qué te pasa?","Tú eres desafiante, ¿qué está sucediendo?"
2367,Puelche,"No me quiero ir a la playa, el puelche está muy fuerte.","No me quiero ir a la playa, el viento está muy fuerte."


## Notas

- **Columnas del CSV:** `clave`, `termino`, `def_id` (id de la definición que el modelo dice ilustrar, `None` si no la indicó bien), `frase_chilena`, `frase_neutralizada`, `contiene_clave`, `neutral_sin_clave`, `ronda`, `modelo`.
- **`contiene_clave`:** la frase chilena incluye la expresión o una variante (acepta conjugaciones comparando los primeros 4 caracteres de cada palabra). **`neutral_sin_clave`:** la frase neutral no incluye la expresión. El archivo `_filtrado.csv` conserva solo los pares que cumplen ambas.
- **Se descartan siempre:** pares con una frase vacía o de largo fuera de 3 a 45 palabras, pares con las dos frases iguales, copias de los ejemplos del diccionario y frases chilenas repetidas.
- **Retomar:** si se corta la ejecución, vuelve a correr la celda de ejecución; las expresiones ya guardadas se saltan.
- **Calidad:** el modelo puede neutralizar mal o conservar chilenismos en la frase neutral. Conviene revisar a mano una muestra (la última celda muestra 10) antes de usar el dataset para entrenar.